In [ ]:
import json
from pathlib import Path
cols = spark.table("proc_pvsyst").columns
rep={"proc_pvsyst_cols": cols, "tiene_E_Grid": "E_Grid" in cols}
# sumas de columnas numericas de proc_pvsyst PL2 en el periodo
import pyspark.sql.functions as F
df = spark.sql("SELECT * FROM proc_pvsyst WHERE plant_code='PL2' AND Date BETWEEN TIMESTAMP'2026-06-02' AND TIMESTAMP'2026-06-12'")
sums={}
for c,t in df.dtypes:
    if t in ('double','float','int','bigint','long') and c not in ('plant_code',):
        try: sums[c]=float(df.agg(F.sum(c)).first()[0] or 0)
        except: pass
rep["sumas_periodo"]=sums
rep["n_pvsyst"]=df.count()
# measured: E_GRID_01 de scada_proc validos
m=spark.sql("SELECT SUM(E_GRID_01) s, COUNT(*) n FROM scada_proc WHERE plant_code='PL2' AND all_valid=1 AND Date BETWEEN TIMESTAMP'2026-06-02' AND TIMESTAMP'2026-06-12'").first()
rep["scada_E_GRID_01_sum_validos"]=float(m["s"] or 0); rep["n_validos"]=m["n"]
Path("/lakehouse/default/Files/PERFORMANCE/_pvdiag.json").write_text(json.dumps(rep,indent=2,default=str))
print(json.dumps(rep,indent=2,default=str))
import notebookutils; notebookutils.notebook.exit("ok")
